In [23]:
import pandas as pd
from pathlib import Path
import numpy as np

In [24]:
stock_list = pd.read_csv('Eastmoney_report_pdf_download/HS300.csv',encoding='utf-8')


In [25]:
stock_list

,序号,股票代码,股票简称,主营行业
0,1,688981,中芯国际,电子设备
1,2,688599,天合光能,电气设备
2,3,688506,百利天恒,医药生物
3,4,688472,阿特斯,电子设备
4,5,688396,华润微,电子设备
...,...,...,...,...
295,296,157,中联重科,机械设备
296,297,100,TCL科技,电子设备
297,298,63,中兴通讯,信息技术
298,299,2,万科A,房地产


In [26]:
df_full_trading_data = pd.DataFrame()

In [27]:
import pandas as pd
from pathlib import Path

def process_single_stock(code, folder="trading_data"):
    """
    处理单个股票的CSV文件，返回一个DataFrame
    - 输入: 股票代码 (int或str)，文件夹路径
    - 输出: DataFrame，索引为交易日期，列名为股票代码
    """
    if code == '沪深300':
        codes_padded = code
    else:
        codes_padded = str(code).zfill(6)  # 补齐为6位
    path = Path(folder) / f"{codes_padded}.csv"

    if not path.exists():
        print(f"文件不存在: {path}")
        return None

    temp = pd.read_csv(path, encoding='utf-8')
    temp = temp[['交易日期', '涨跌幅(%)']]
    # temp['交易日期'] = pd.to_datetime(temp['交易日期'])
    temp = temp.drop_duplicates(subset=['交易日期'])  # 去重
    temp = temp.rename(columns={'涨跌幅(%)': codes_padded})
    temp = temp.set_index('交易日期')

    return temp


In [28]:
process_single_stock('000767')

,000767
交易日期,
20251205,1.3986
20251204,-0.6944
20251203,0.6993
20251202,0.3509
20251201,0.7067
...,...
20000110,1.7300
20000107,0.5300
20000106,3.2800


In [29]:
all_data = []

for code in stock_list['股票代码']:
    df = process_single_stock(code, folder="trading_data")
    if df is not None:
        all_data.append(df)
all_data.append(process_single_stock('沪深300'))
# 拼接成大表
big_table = pd.concat(all_data, axis=1, join='outer').reset_index().sort_values(by='交易日期')
big_table['交易日期'] = pd.to_datetime(big_table['交易日期'], format='%Y%m%d')
big_table = big_table.dropna(subset=['交易日期'])
big_table.to_csv("all_stock_returns.csv", index=False, encoding="utf-8")


In [30]:
import pandas as pd

# 转换所有收益列为数值
for col in big_table.columns:
	if col != '交易日期':
		big_table[col] = pd.to_numeric(big_table[col], errors='coerce')

big_table = big_table.fillna(0)


# 转换为日收益率 (小数)
returns = big_table.drop(columns=['交易日期']) / 100.0

# 计算日复利因子
factors = 1 + returns
factors['交易日期'] = big_table['交易日期']

print(factors.shape)



(6283, 302)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_72924\1537288333.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  factors['交易日期'] = big_table['交易日期']


In [31]:
print(factors['交易日期'])


6238   2000-01-04
6237   2000-01-05
6236   2000-01-06
6235   2000-01-07
6234   2000-01-10
          ...    
4      2025-12-01
3      2025-12-02
2      2025-12-03
1      2025-12-04
0      2025-12-05
Name: 交易日期, Length: 6283, dtype: datetime64[ns]


In [32]:
# 将交易日期转换为季度周期
factors['交易日期'] = pd.to_datetime(factors['交易日期']).dt.to_period("Q")

print(factors)
# # 按季度聚合：乘积 - 1

quarterly_returns = (factors.groupby('交易日期').prod() - 1).reset_index()

        688981    688599    688506    688472    688396    688303    688271  \
6238  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
6237  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
6236  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
6235  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
6234  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
...        ...       ...       ...       ...       ...       ...       ...   
4     1.007042  0.977336  0.986008  0.930814  1.096920  1.013603  0.986476   
3     0.987762  0.986425  0.987332  1.005621  0.969566  0.961459  0.980274   
2     0.987788  0.974771  0.979049  0.977019  1.079384  0.993558  1.001321   
1     1.027862  0.974706  1.009207  0.977750  1.017824  0.978746  0.999922   
0     0.995206  1.027761  0.997041  1.035761  0.973641  1.009569  1.013502   

        688256    688223    688187  ...    000333    000301    

In [33]:
print(quarterly_returns)


       交易日期    688981    688599    688506    688472    688396    688303  \
0    2000Q1  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
1    2000Q2  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
2    2000Q3  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
3    2000Q4  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
4    2001Q1  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
..      ...       ...       ...       ...       ...       ...       ...   
99   2024Q4  0.577263 -0.139157 -0.020386 -0.100928  0.000845 -0.078631   
100  2025Q1 -0.055909 -0.127979  0.211650 -0.225319 -0.050645 -0.196355   
101  2025Q2 -0.012988 -0.136660  0.274674 -0.059611  0.054050  0.098967   
102  2025Q3  0.589316  0.195455  0.267392  0.483416  0.177905  0.347561   
103  2025Q4 -0.185186 -0.019570 -0.030244  0.185268 -0.048617 -0.045248   

       688271    688256    688223  ...    000338    000333    000301  \
0    0.000000  0.000000  0.

In [34]:
quarterly_excess_returns = quarterly_returns.drop(columns=['交易日期']).sub(
    quarterly_returns['沪深300'], axis=0
)
quarterly_excess_returns['交易日期'] = quarterly_returns['交易日期']

In [35]:
quarterly_excess_returns

,688981,688599,688506,688472,688396,688303,688271,688256,688223,688187,...,000333,000301,000166,000157,000100,000063,000002,000001,沪深300,交易日期
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.191650,0.375095,0.052625,0.0,2000Q1
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.940137,0.000000,0.000000,0.000000,0.040490,0.064532,-0.013114,0.0,2000Q2
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-0.048142,0.000000,0.000000,0.000000,0.225782,-0.074695,-0.064696,0.0,2000Q3
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.085481,0.000000,0.938983,0.000000,0.047119,0.131582,-0.037339,0.0,2000Q4
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.007599,0.000000,-0.056387,0.000000,-0.052326,0.081252,0.117094,0.0,2001Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,0.577263,-0.139157,-0.020386,-0.100928,0.000845,-0.078631,-0.011560,1.275557,-0.192964,-0.097044,...,-0.011047,-0.115304,-0.061188,-0.046174,0.098254,0.296952,-0.253090,-0.020811,0.0,2024Q4
100,-0.055909,-0.127979,0.211650,-0.225319,-0.050645,-0.196355,-0.034887,-0.053193,-0.088611,-0.013566,...,0.043601,0.009744,-0.078508,0.040111,-0.115308,-0.152968,-0.028923,-0.037608,0.0,2025Q1
101,-0.012988,-0.136660,0.274674,-0.059611,0.054050,0.098967,0.047135,-0.034511,-0.199076,-0.097737,...,-0.035807,0.004829,0.018255,-0.038568,-0.026966,-0.031889,-0.089364,0.105526,0.0,2025Q2
102,0.589316,0.195455,0.267392,0.483416,0.177905,0.347561,0.188261,1.202814,0.071288,0.262268,...,0.006371,0.141658,0.071661,0.150938,0.006293,0.404741,0.073207,-0.060481,0.0,2025Q3


In [36]:
import numpy as np
import pandas as pd

# 只保留数值列
numeric_df = quarterly_excess_returns.select_dtypes(include=[np.number])

# 展平所有数值，计算分位点
flattened = numeric_df.values.flatten()
flattened = flattened[~np.isnan(flattened)]
flattened = flattened[flattened != 0] # 关键：忽略 0 值
q = pd.Series(flattened).quantile([0.2, 0.4, 0.6, 0.8])

print(q)

# 定义标签函数
def label_5class(x):
    if pd.isna(x):
        return np.nan
    elif x < q[0.2]:
        return 0
    elif x < q[0.4]:
        return 1
    elif x < q[0.6]:
        return 2
    elif x < q[0.8]:
        return 3
    else:
        return 4

# 应用到数值列
labels_5class = numeric_df.applymap(label_5class)

# 拼回交易日期
labels_5class['交易日期'] = quarterly_excess_returns['交易日期']


0.2   -0.118124
0.4   -0.025112
0.6    0.054669
0.8    0.188185
dtype: float64


C:\Users\Administrator\AppData\Local\Temp\ipykernel_72924\1372170503.py:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  labels_5class = numeric_df.applymap(label_5class)


In [37]:
import pandas as pd

# 假设 labels_5class 已经生成，并且包含交易日期列
# 我们只统计标签列，不包括交易日期
label_counts = labels_5class.drop(columns=['交易日期']).stack().value_counts(normalize=True)

# 转换成百分比形式
label_percentages = (label_counts * 100).round(2)

print("各标签百分比：")
print(label_percentages)


各标签百分比：
2    49.99
4    12.50
0    12.50
3    12.50
1    12.50
Name: proportion, dtype: float64


In [38]:
labels_5class.to_csv("labels.csv", index=False, encoding="utf-8")
